# 01 — Exploratory Data Analysis: Longitudinal Topology, Population Drift & Complexity Justification

## Executive Summary
This notebook performs a rigorous statistical and visual audit of the Silver Layer. The analysis axes are:

| # | Analysis Axis | Business Question |
|---|---|---|
| 1 | Temporal Integrity | Do all 20,931 HCPs have exactly 86 weeks of data? |
| 2 | Distributional Drift | Are the 11,899 labeled HCPs representative of the 9,032 unlabeled ones? |
| 3 | Sparsity Profiling | How sparse is the data? Are most features dominated by zeros? |
| 4 | Macro Longitudinal Trends | Is there global seasonality or market drift over the 86-week window? |
| 5 | Micro Deep-Dive | How volatile is an individual HCP's behavior over time? |
| 6 | Segment Profiling | How do the ATSEG segments (A, B, C) differ in prescribing behavior? |
| 7 | Spatial Asymmetry & KOLs | Do mega-prescribers distort the distributions? |

**Constraint:** This notebook is strictly *read-only*. No DataFrame is exported and no data is altered.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
import logging

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s — %(levelname)s — %(message)s')

# Professional visualization defaults
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 120
})

# Column registries
PRESCRIPTION_COLS = [
    'UC_TRX', 'ORAL_TRX', 'IL23_TRX', 'BRAND1_TRX', 'BRAND2_TRX',
    'UC_NRX', 'ORAL_NRX', 'IL23_NRX', 'BRAND1_NRX', 'BRAND2_NRX',
    'BRAND1_NBRX', 'BRAND2_NBRX', 'ORAL_NBRX', 'IL23_NBRX'
]
MARKETING_COLS = ['RTE', 'SAMPLES', 'COPAY', 'DIRECTMAIL', 'SPK', 'DETAILS']

logging.info("Environment initialized.")

---
## 1. Data Loading & Temporal Integrity Check
We load the Silver Layer and verify that the 86-week longitudinal structure was perfectly preserved after the cleaning step.

In [ ]:
SILVER_PATH = "data/silver_layer_longitudinal.parquet"

df = pd.read_parquet(SILVER_PATH)
logging.info(f"Silver Layer loaded: {df.shape[0]:,} rows x {df.shape[1]} columns.")

# Filter column registries to only those present in data
PRESCRIPTION_COLS = [c for c in PRESCRIPTION_COLS if c in df.columns]
MARKETING_COLS = [c for c in MARKETING_COLS if c in df.columns]

print(f"Unique HCPs: {df['NUEVO_ID'].nunique():,}")
print(f"Unique Weeks: {df['WEEK_ID'].nunique()}")
print(f"Week Range: {df['WEEK_ID'].min()} to {df['WEEK_ID'].max()}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Verify sequence lengths
weeks_per_hcp = df.groupby('NUEVO_ID')['WEEK_ID'].count()

fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(weeks_per_hcp, bins=30, color='#5C6BC0', edgecolor='white', ax=ax)
ax.axvline(x=86, color='crimson', linestyle='--', linewidth=2, label='Expected: 86 weeks')
ax.set_title('Distribution of Longitudinal Sequence Lengths per HCP')
ax.set_xlabel('Number of Weeks Tracked')
ax.set_ylabel('HCP Count')
ax.legend()
plt.tight_layout()
plt.show()

intact_ratio = (weeks_per_hcp == 86).mean()
print(f"Temporal Integrity: {intact_ratio:.2%} of HCPs have exactly 86 weeks.")

### Finding: Perfect Temporal Integrity ✅
**100.00%** of the 20,931 HCPs possess exactly 86 weekly records spanning from **2024-01-05** to **2025-08-22**. Every sequence is complete — no imputation or padding is needed. This means our Deep Learning tensors will be uniformly shaped `(N, 86, F)`, simplifying batching and training significantly.

---
## 2. Sparsity Profiling
Before analyzing distributions, we must understand how *sparse* the data is. In pharmaceutical datasets, the majority of HCPs prescribe zero units in most weeks for niche products.

In [ ]:
# Compute zero-percentage for every metric column
all_metric_cols = PRESCRIPTION_COLS + MARKETING_COLS
sparsity_data = []
for col in all_metric_cols:
    zero_pct = (df[col] == 0).mean()
    sparsity_data.append({'Feature': col, 'Zero Percentage': zero_pct, 'Type': 'Prescription' if col in PRESCRIPTION_COLS else 'Marketing'})

sparsity_df = pd.DataFrame(sparsity_data).sort_values('Zero Percentage', ascending=False)

fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#1565C0' if t == 'Prescription' else '#E65100' for t in sparsity_df['Type']]
bars = ax.barh(sparsity_df['Feature'], sparsity_df['Zero Percentage'] * 100, color=colors, edgecolor='white')
ax.axvline(x=90, color='red', linestyle='--', alpha=0.7, label='90% Sparsity Threshold')
ax.set_xlabel('Zero Values (%)')
ax.set_title('Feature Sparsity Profile — Percentage of Zero Values per Column')
ax.legend()

# Annotate percentages
for bar, pct in zip(bars, sparsity_df['Zero Percentage']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{pct:.1%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### Finding: Extreme Sparsity in Brand-Level Metrics ⚠️
The data exhibits massive sparsity, which is a critical architectural consideration:

| Sparsity Tier | Features | Zero % |
|---|---|---|
| **Ultra-Sparse (>99%)** | BRAND1_NBRX, BRAND2_NBRX, BRAND1_NRX, BRAND2_NRX, BRAND1_TRX, SPK, SAMPLES, COPAY | 99.6–99.9% |
| **Very Sparse (90–99%)** | IL23_TRX, IL23_NRX, ORAL_NRX, ORAL_NBRX, IL23_NBRX, BRAND2_TRX, RTE, DIRECTMAIL, DETAILS | 91–98% |
| **Moderate (50–90%)** | UC_NRX, ORAL_TRX | 80–85% |
| **Least Sparse** | UC_TRX | 58% |

**Implication:** `UC_TRX` is the most informative signal (42% non-zero). Brand-level metrics (`BRAND1_*`, `BRAND2_*`) are extremely sparse — most HCPs never prescribe the brand products. This means the model needs to learn from *rare events*, which favors architectures that can attend to non-zero spikes (like 1D-CNNs with dilated convolutions).

---
## 3. Distributional Drift: Labeled vs. Unlabeled Cohorts
The most important question in semi-supervised learning: **Are our 11,899 labeled HCPs representative of the 9,032 unlabeled ones?**

We use two tools:
1. **KDE Plots** for visual overlay comparison.
2. **Population Stability Index (PSI)** for quantitative measurement.

In [ ]:
# Consolidate at HCP level for drift analysis
hcp_level = df.groupby(['NUEVO_ID', 'IS_LABELED'])[PRESCRIPTION_COLS + MARKETING_COLS].mean().reset_index()

labeled_count = hcp_level[hcp_level['IS_LABELED'] == True].shape[0]
unlabeled_count = hcp_level[hcp_level['IS_LABELED'] == False].shape[0]
print(f"Labeled HCPs: {labeled_count:,} ({labeled_count / len(hcp_level):.1%})")
print(f"Unlabeled HCPs: {unlabeled_count:,} ({unlabeled_count / len(hcp_level):.1%})")

In [ ]:
# KDE Overlay Plots for key metrics
drift_metrics = ['UC_TRX', 'ORAL_TRX', 'BRAND1_NBRX', 'DETAILS']
drift_metrics = [c for c in drift_metrics if c in hcp_level.columns]

fig, axes = plt.subplots(1, len(drift_metrics), figsize=(6 * len(drift_metrics), 5))
if len(drift_metrics) == 1:
    axes = [axes]

for ax, metric in zip(axes, drift_metrics):
    for label_val, color, label_text in [(True, '#2196F3', 'Labeled'), (False, '#FF9800', 'Unlabeled')]:
        subset = hcp_level.loc[hcp_level['IS_LABELED'] == label_val, metric].dropna()
        if len(subset) > 1:
            sns.kdeplot(subset, ax=ax, color=color, fill=True, alpha=0.35,
                        linewidth=2, label=f'{label_text} (n={len(subset):,})')
    ax.set_title(f'Drift: {metric}')
    ax.set_xlabel(f'Mean {metric} per HCP')
    ax.legend()

plt.suptitle('Kernel Density Estimation — Labeled vs. Unlabeled Cohorts', fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
def compute_psi(reference, current, n_bins=10):
    """
    Computes the Population Stability Index (PSI).

    Interpretation:
      < 0.10  : No significant shift
      0.10-0.25 : Moderate shift — investigate
      > 0.25  : Severe shift — populations are fundamentally different
    """
    reference = np.array(reference, dtype=float)
    current = np.array(current, dtype=float)

    breakpoints = np.percentile(reference, np.linspace(0, 100, n_bins + 1))
    breakpoints = np.unique(breakpoints)

    ref_counts = np.histogram(reference, bins=breakpoints)[0].astype(float)
    cur_counts = np.histogram(current, bins=breakpoints)[0].astype(float)

    ref_proportions = ref_counts / ref_counts.sum()
    cur_proportions = cur_counts / cur_counts.sum()

    epsilon = 1e-8
    ref_proportions = np.clip(ref_proportions, epsilon, None)
    cur_proportions = np.clip(cur_proportions, epsilon, None)

    return np.sum((cur_proportions - ref_proportions) * np.log(cur_proportions / ref_proportions))

# Compute PSI for key metrics
reference_cohort = hcp_level[hcp_level['IS_LABELED'] == False]
current_cohort = hcp_level[hcp_level['IS_LABELED'] == True]

psi_results = []
for metric in PRESCRIPTION_COLS + MARKETING_COLS:
    ref_vals = reference_cohort[metric].dropna().values
    cur_vals = current_cohort[metric].dropna().values
    if len(ref_vals) > 10 and len(cur_vals) > 10:
        psi = compute_psi(ref_vals, cur_vals, n_bins=10)
        psi_results.append({'Metric': metric, 'PSI': round(psi, 4)})

psi_df = pd.DataFrame(psi_results).sort_values('PSI', ascending=False)
psi_df['Interpretation'] = psi_df['PSI'].apply(
    lambda x: '🔴 SEVERE SHIFT' if x > 0.25 else ('🟡 MODERATE' if x > 0.10 else '🟢 STABLE')
)
print("\n" + "=" * 60)
print("  POPULATION STABILITY INDEX (PSI) REPORT")
print("=" * 60)
print(psi_df.to_string(index=False))

### Finding: Significant Population Drift Detected 🔴

The PSI analysis reveals a **major finding** that directly impacts model design:

| Metric | PSI | Status |
|---|---|---|
| **UC_TRX** | **1.7753** | 🔴 Severe Shift |
| **ORAL_TRX** | **0.7096** | 🔴 Severe Shift |
| **DETAILS** | **0.6971** | 🔴 Severe Shift |
| BRAND1_NBRX | 0.0000 | 🟢 Stable |
| BRAND1_TRX | 0.0000 | 🟢 Stable |

**What this means:** The labeled cohort (11,899 HCPs) prescribes significantly **more** than the unlabeled cohort (9,032 HCPs) in aggregate UC and ORAL categories. This is expected because Pfizer's labeling process likely targeted **clinically active** gastroenterologists, not low-volume prescribers.

**Impact on modeling:**
- A model trained only on labeled data may **overestimate** prescribing activity when applied to unlabeled HCPs.
- The brand-level metrics (BRAND1, BRAND2) show zero drift because they are ultra-sparse in both populations.
- **Mitigation strategy:** Semi-supervised learning and careful pseudo-labeling with conservative confidence thresholds will be critical in Phases V–VI of the pipeline.

---
## 4. Marketing Exposure Asymmetry
Before analyzing temporal trends, let's quantify whether labeled vs. unlabeled HCPs receive different levels of marketing engagement.

In [ ]:
# Compare average marketing exposure by label status
hcp_mkt = df.groupby(['NUEVO_ID', 'IS_LABELED'])[MARKETING_COLS].mean().reset_index()
mkt_comparison = hcp_mkt.groupby('IS_LABELED')[MARKETING_COLS].mean()

fig, ax = plt.subplots(figsize=(12, 6))
mkt_comparison.T.plot(kind='bar', ax=ax, color=['#FF9800', '#2196F3'], edgecolor='white', width=0.7)
ax.set_title('Average Weekly Marketing Exposure: Labeled vs. Unlabeled HCPs')
ax.set_ylabel('Mean Weekly Value')
ax.set_xlabel('Marketing Channel')
ax.legend(['Unlabeled (n=9,032)', 'Labeled (n=11,899)'], loc='upper right')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### Finding: Labeled HCPs Receive 2–6x More Marketing ⚠️
The labeled cohort receives substantially more marketing engagement across all channels:

| Channel | Unlabeled Avg | Labeled Avg | Ratio |
|---|---|---|---|
| DETAILS | 0.0244 | 0.0806 | **3.3x** |
| DIRECTMAIL | 0.0186 | 0.0688 | **3.7x** |
| RTE | 0.0094 | 0.0494 | **5.3x** |
| COPAY | 0.0008 | 0.0050 | **6.2x** |
| SAMPLES | 0.0008 | 0.0028 | **3.7x** |

**Interpretation:** This is not a flaw in the data — it is a reflection of real-world commercial strategy. Pharma sales teams focus marketing resources on HCPs they have already profiled (labeled). This correlation between labeling and marketing exposure must be accounted for in the model to avoid confounding.

---
## 5. ATSEG Segment Profiling
The target variable `ATSEG` has three segments: `SEG_A`, `SEG_B`, and `SEG_C`. Let's understand how they differ in prescribing behavior.

In [ ]:
# Segment distribution at HCP level
labeled_hcps_df = df[df['IS_LABELED'] == True]
hcp_segment = labeled_hcps_df.groupby('NUEVO_ID').agg(
    ATSEG=('ATSEG', 'first'),
    SPECIALTY=('SPECIALTY', 'first'),
    AVG_UC_TRX=('UC_TRX', 'mean'),
    AVG_ORAL_TRX=('ORAL_TRX', 'mean'),
    AVG_IL23_TRX=('IL23_TRX', 'mean'),
    AVG_DETAILS=('DETAILS', 'mean')
).reset_index()

print("ATSEG Distribution (HCP-level):")
print(hcp_segment['ATSEG'].value_counts().to_string())
print(f"\nTotal Labeled HCPs: {len(hcp_segment):,}")

In [ ]:
# Segment-level prescribing behavior comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
segment_palette = {'SEG_A': '#4CAF50', 'SEG_B': '#FF9800', 'SEG_C': '#F44336'}

for ax, metric in zip(axes, ['AVG_UC_TRX', 'AVG_ORAL_TRX', 'AVG_IL23_TRX']):
    sns.boxplot(data=hcp_segment, x='ATSEG', y=metric,
                order=['SEG_A', 'SEG_B', 'SEG_C'], palette=segment_palette,
                fliersize=2, ax=ax)
    ax.set_title(f'{metric} by ATSEG Segment')
    ax.set_ylabel(f'Mean Weekly {metric}')

plt.suptitle('Prescribing Intensity by Target Segment (Labeled HCPs Only)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print summary table
print("\nSegment Summary Statistics (Mean Weekly UC_TRX):")
print(hcp_segment.groupby('ATSEG')['AVG_UC_TRX'].describe().round(4).to_string())

### Finding: Clear Segment Separation in Prescribing Volume ✅

| Segment | Count | Mean UC_TRX/week | Median UC_TRX/week | Interpretation |
|---|---|---|---|---|
| **SEG_A** | 6,406 (54%) | 0.1713 | 0.1198 | Low-volume prescribers |
| **SEG_B** | 3,349 (28%) | 0.5174 | 0.3504 | Medium-volume prescribers |
| **SEG_C** | 2,144 (18%) | 0.7111 | 0.3842 | High-volume prescribers |

The segments show a clear gradient in prescribing intensity: **SEG_C > SEG_B > SEG_A**. This validates the target variable — the segmentation is clinically meaningful and not random. However, there is significant overlap in the distributions (note the similar medians of B and C), meaning classification will require nuanced features beyond simple averages.

In [ ]:
# Specialty x Segment cross-tabulation
cross_tab = pd.crosstab(hcp_segment['SPECIALTY'], hcp_segment['ATSEG'], margins=True)
cross_tab_pct = pd.crosstab(hcp_segment['SPECIALTY'], hcp_segment['ATSEG'], normalize='index').round(3)

print("Specialty x ATSEG Cross-Tabulation (Counts):")
print(cross_tab.to_string())
print("\nSpecialty x ATSEG Cross-Tabulation (Row %):")
print(cross_tab_pct.to_string())

### Finding: Gastroenterologists Dominate the Labeled Set
The specialty `GE` (Gastroenterology) represents **98.2%** of all labeled HCPs (11,680 out of 11,899). Other specialties (IM, PHA, NRP, GPFM) are marginal minorities. This extreme class imbalance means the model will primarily learn gastroenterologist behavior, which is appropriate for the UC (Ulcerative Colitis) therapeutic area.

---
## 6. Global Longitudinal Trends (Macro View)
Let's examine the market-level temporal evolution to detect seasonality, promotional campaign effects, or structural drift.

In [ ]:
trend_metrics = [c for c in ['UC_TRX', 'ORAL_TRX', 'BRAND1_NBRX', 'DETAILS'] if c in df.columns]

# Assign numeric week index for proper plotting
week_order = sorted(df['WEEK_ID'].unique())
week_map = {w: i + 1 for i, w in enumerate(week_order)}
df['WEEK_NUM'] = df['WEEK_ID'].map(week_map)

global_weekly_means = df.groupby('WEEK_NUM')[trend_metrics].mean()

fig, axes = plt.subplots(len(trend_metrics), 1, figsize=(16, 4 * len(trend_metrics)), sharex=True)
if len(trend_metrics) == 1:
    axes = [axes]

colors = ['#1565C0', '#2E7D32', '#F57F17', '#AD1457']

for ax, metric, color in zip(axes, trend_metrics, colors):
    ax.plot(global_weekly_means.index, global_weekly_means[metric],
            color=color, linewidth=2.2, label=metric)
    ax.fill_between(global_weekly_means.index, 0, global_weekly_means[metric],
                    alpha=0.12, color=color)
    ax.set_ylabel('Population Mean')
    ax.legend(loc='upper right', fontsize=11)
    ax.set_title(f'Global Weekly Trend: {metric}', fontsize=13)

axes[-1].set_xlabel('Week Number (1–86, Chronological)')
plt.suptitle('Market-Level Longitudinal Trends over 86 Weeks (Jan 2024 – Aug 2025)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Single HCP Deep-Dive: The Case for Deep Sequence Learning

### Why can't we just "average" everything?

We isolate a single labeled HCP with the highest temporal volatility in `UC_TRX`. This exercise is the **philosophical cornerstone** of the entire pipeline:

- If behavior were flat, a simple mean would suffice.
- But real physicians change patterns: they adopt brands, abandon them, respond to rep visits, and revert to baseline.

These **regime changes, spikes, and drops** are exactly the signal that a Deep Sequence Learning model (1D-CNN, LSTM, Transformer) can capture — and that a summary statistics table would completely destroy.

In [ ]:
# Identify the most volatile labeled HCP
labeled_hcp_ids = df.loc[df['IS_LABELED'] == True, 'NUEVO_ID'].unique()

labeled_volatility = (
    df[df['NUEVO_ID'].isin(labeled_hcp_ids)]
    .groupby('NUEVO_ID')['UC_TRX']
    .std()
    .sort_values(ascending=False)
)

target_hcp_id = labeled_volatility.index[0]
target_hcp_std = labeled_volatility.iloc[0]

# Print context about this HCP
target_info = df[df['NUEVO_ID'] == target_hcp_id].iloc[0]
print(f"Selected HCP for Deep-Dive: NUEVO_ID = {target_hcp_id}")
print(f"Volatility (Std Dev of UC_TRX): {target_hcp_std:.4f}")
print(f"Specialty: {target_info['SPECIALTY']}, State: {target_info['STATE']}, Segment: {target_info['ATSEG']}")
print(f"This HCP ranks #1 out of {len(labeled_volatility):,} labeled HCPs in temporal volatility.")

In [ ]:
# Extract the full 86-week timeline
single_hcp = df[df['NUEVO_ID'] == target_hcp_id].sort_values('WEEK_NUM').copy()

# Define plot layers
rx_plot_cols = [c for c in ['UC_TRX', 'ORAL_TRX', 'BRAND1_NBRX'] if c in single_hcp.columns]
mkt_plot_cols = [c for c in ['DETAILS', 'SAMPLES', 'RTE'] if c in single_hcp.columns]

fig, (ax_rx, ax_mkt) = plt.subplots(2, 1, figsize=(18, 11), sharex=True,
                                     gridspec_kw={'height_ratios': [3, 2]})

# Top Panel: Prescriptions
rx_colors = ['#1565C0', '#E65100', '#2E7D32']
for col, color in zip(rx_plot_cols, rx_colors):
    ax_rx.plot(single_hcp['WEEK_NUM'], single_hcp[col],
              marker='o', markersize=3.5, linewidth=1.8, color=color, label=col, alpha=0.85)

ax_rx.set_ylabel('Prescription Volume', fontsize=12)
ax_rx.set_title(
    f'HCP #{target_hcp_id} — 86-Week Longitudinal Deep-Dive\n'
    f'(Specialty: {target_info["SPECIALTY"]}, Segment: {target_info["ATSEG"]}, '
    f'UC_TRX StdDev: {target_hcp_std:.2f})',
    fontsize=14, fontweight='bold'
)
ax_rx.legend(loc='upper right', framealpha=0.9)
ax_rx.grid(axis='y', alpha=0.3)

# Bottom Panel: Marketing
mkt_colors = ['#7B1FA2', '#00838F', '#BF360C']
if mkt_plot_cols:
    bar_width = 0.3
    x_positions = single_hcp['WEEK_NUM'].values
    for i, (col, color) in enumerate(zip(mkt_plot_cols, mkt_colors)):
        offset = (i - len(mkt_plot_cols) / 2) * bar_width
        ax_mkt.bar(x_positions + offset, single_hcp[col].values,
                   width=bar_width, color=color, alpha=0.65, label=col)

ax_mkt.set_xlabel('Week Number (1–86)', fontsize=12)
ax_mkt.set_ylabel('Marketing Contacts', fontsize=12)
ax_mkt.set_title(f'HCP #{target_hcp_id} — Marketing Exposure Timeline', fontsize=13, fontweight='bold')
ax_mkt.legend(loc='upper right', framealpha=0.9)
ax_mkt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary stats for the selected HCP
print("\nSelected HCP Summary Statistics:")
display_cols = [c for c in ['UC_TRX', 'ORAL_TRX', 'BRAND1_NBRX', 'DETAILS', 'SAMPLES'] if c in single_hcp.columns]
print(single_hcp[display_cols].describe().round(4).to_string())

### Finding: Individual HCP Behavior Is Radically Non-Stationary ⚡

**HCP #12981** (Gastroenterologist, SEG_B) is our most volatile labeled prescriber with a UC_TRX standard deviation of **4.45** (against a population mean of 0.24). Key observations:

1. **Regime Changes:** The UC_TRX signal oscillates dramatically — from 0 prescriptions in early weeks to peaks of ~14 units, then dropping back to near-zero. These are not random fluctuations; they represent real clinical behavior shifts.

2. **The ORAL_TRX Co-Movement:** When UC_TRX spikes, ORAL_TRX follows with a correlated but smaller spike (r ≈ 0.87 at population level). This cross-variable temporal correlation is exactly what a 1D-CNN can learn from multi-channel input.

3. **Marketing Response Windows:** The DETAILS contacts (rep visits) are sparse but temporally concentrated. A sequence model can learn whether prescription spikes *follow* marketing events (causal lag detection).

4. **The Fallacy of the Mean:** This HCP's 86-week average is 4.62 UC_TRX/week. But this single number hides the fact that ~35% of his weeks have ZERO prescriptions, while other weeks exceed 13. **A model that only sees "4.62" would misclassify this doctor's true behavioral pattern.**

**Conclusion:** This Deep-Dive validates the architectural decision to preserve the 86-week granularity and feed complete sequences into the Deep Sequence Learning model (Phase III). Averaging destroys signal. Sequences preserve it.

---
## 8. Correlation Structure Among Prescription Metrics

In [ ]:
# HCP-level correlation matrix
top_rx_cols = [c for c in ['UC_TRX', 'ORAL_TRX', 'IL23_TRX', 'BRAND1_TRX', 'BRAND2_TRX'] if c in df.columns]
hcp_means_corr = df.groupby('NUEVO_ID')[top_rx_cols].mean()

corr_matrix = hcp_means_corr.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlBu_r',
            mask=mask, square=True, linewidths=1, vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlation Matrix: Prescription Metrics (HCP-Level Means)', fontweight='bold')
plt.tight_layout()
plt.show()

### Finding: Strong Collinearity Between UC and ORAL Prescriptions

| Pair | Correlation | Implication |
|---|---|---|
| UC_TRX ↔ ORAL_TRX | **0.869** | Very high — oral therapies track total UC prescriptions closely |
| UC_TRX ↔ IL23_TRX | **0.763** | High — IL23 biologics correlate with overall UC activity |
| BRAND1_TRX ↔ ORAL_TRX | **0.356** | Moderate — brand adoption is partially driven by overall oral activity |
| BRAND1_TRX ↔ BRAND2_TRX | **0.267** | Weak — the two brands are *not* substitutes for each other |

**Implication for modeling:** The high correlation between UC_TRX, ORAL_TRX, and IL23_TRX means there is redundant information. A 1D-CNN's convolutional filters will naturally learn shared representations across these correlated channels, reducing effective dimensionality.

---
## 9. Spatial Asymmetry & KOL Detection
A handful of Key Opinion Leaders (KOLs) concentrate a disproportionate share of total prescribing volume. We use log-scale visualization to prevent these outliers from distorting the view.

In [ ]:
# Aggregate total UC_TRX per HCP across all 86 weeks
hcp_total_volume = df.groupby(['NUEVO_ID', 'SPECIALTY'])['UC_TRX'].sum().reset_index()
hcp_total_volume.rename(columns={'UC_TRX': 'TOTAL_UC_TRX'}, inplace=True)

# Keep only specialties with meaningful counts
spec_counts = hcp_total_volume['SPECIALTY'].value_counts()
valid_specs = spec_counts[spec_counts >= 20].index.tolist()
filtered = hcp_total_volume[hcp_total_volume['SPECIALTY'].isin(valid_specs)]

fig, ax = plt.subplots(figsize=(14, 7))
order = filtered.groupby('SPECIALTY')['TOTAL_UC_TRX'].median().sort_values(ascending=False).index
sns.boxplot(data=filtered, x='SPECIALTY', y='TOTAL_UC_TRX',
            order=order, palette='viridis', fliersize=2, ax=ax)
ax.set_yscale('log')
ax.set_title('KOL Detection: Total UC_TRX by Specialty (Log Scale, 86 Weeks)', fontweight='bold')
ax.set_xlabel('Medical Specialty')
ax.set_ylabel('Total UC_TRX per HCP (Log Scale)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

# Quantify extreme outliers
p99 = hcp_total_volume['TOTAL_UC_TRX'].quantile(0.99)
kol_count = (hcp_total_volume['TOTAL_UC_TRX'] > p99).sum()
print(f"\nKOL Threshold (P99): {p99:.2f} total UC_TRX over 86 weeks")
print(f"Potential KOLs above P99: {kol_count} HCPs ({kol_count / len(hcp_total_volume):.2%})")

---
## 10. Demographic Overview

In [ ]:
# Age distribution
hcp_demographics = df.groupby('NUEVO_ID').agg(
    SPECIALTY=('SPECIALTY', 'first'),
    AGE_RANGE=('AGE_RANGE', 'first'),
    STATE=('STATE', 'first'),
    IS_LABELED=('IS_LABELED', 'first')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Specialty
spec_counts = hcp_demographics['SPECIALTY'].value_counts()
axes[0].barh(spec_counts.index, spec_counts.values, color='#5C6BC0', edgecolor='white')
axes[0].set_title('HCP Distribution by Specialty')
axes[0].set_xlabel('Count')
for i, v in enumerate(spec_counts.values):
    axes[0].text(v + 50, i, f'{v:,}', va='center', fontsize=9)

# Age Range
age_counts = hcp_demographics['AGE_RANGE'].value_counts()
axes[1].barh(age_counts.index, age_counts.values, color='#26A69A', edgecolor='white')
axes[1].set_title('HCP Distribution by Age Range')
axes[1].set_xlabel('Count')
for i, v in enumerate(age_counts.values):
    axes[1].text(v + 50, i, f'{v:,}', va='center', fontsize=9)

# State (top 8)
state_counts = hcp_demographics['STATE'].value_counts().head(8)
axes[2].barh(state_counts.index.astype(str), state_counts.values, color='#FF7043', edgecolor='white')
axes[2].set_title('HCP Distribution by State (Top 8)')
axes[2].set_xlabel('Count')
for i, v in enumerate(state_counts.values):
    axes[2].text(v + 50, i, f'{v:,}', va='center', fontsize=9)

plt.suptitle('Demographic Profile of the HCP Population (n=20,931)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Finding: Demographic Profile

| Dimension | Distribution |
|---|---|
| **Specialty** | 79% GE (Gastroenterology), 12% IM (Internal Medicine), remainder split across PHA, NRP, GPFM |
| **Age Range** | Roughly balanced thirds: (1960–1980] 37%, (1980–2000] 33%, (1940–1960] 30% |
| **State** | State "1" dominates with 62% of HCPs, followed by States 5, 2, and 8 |

In [ ]:
# Clean up temporary column
df.drop(columns=['WEEK_NUM'], inplace=True, errors='ignore')

---
## Summary of Key Findings

| # | Finding | Severity | Implication for Pipeline |
|---|---|---|---|
| 1 | **100% temporal integrity** — all 20,931 HCPs have 86 weeks | ✅ Green | Tensors will be uniformly shaped (N, 86, F) |
| 2 | **Severe population drift** — PSI > 1.7 for UC_TRX | 🔴 Critical | Labeled HCPs prescribe 3–5x more than unlabeled; semi-supervised approach essential |
| 3 | **Labeled HCPs receive 3–6x more marketing** | ⚠️ Warning | Marketing exposure is confounded with label status |
| 4 | **Extreme sparsity** — brand metrics are 99%+ zeros | ⚠️ Warning | Model must learn from rare events; zero-inflated architectures may help |
| 5 | **Clear ATSEG gradient** — SEG_C > SEG_B > SEG_A in volume | ✅ Green | Target variable is clinically meaningful |
| 6 | **Individual behavior is radically non-stationary** | ⚡ Insight | Averaging destroys signal; Deep Sequence Learning is justified |
| 7 | **High correlation** between UC_TRX, ORAL_TRX, IL23_TRX | ⚠️ Warning | Redundant features; CNN filters will compress these naturally |
| 8 | **GE specialty dominates** (79% of population) | ℹ️ Note | Model learns primarily gastroenterologist patterns |
| 9 | **KOLs exist** at P99 threshold | ⚠️ Warning | Consider winsorization or separate treatment in Gold Layer |